# BiLSTM baseline on LIAR (statement text only)
This notebook trains and evaluates the BiLSTM baseline from scratch. It needs only a CPU (no GPU): the full run took about 4 minutes on a fast laptop (free Colab CPUs can be slower). Runtime > Run all.

**Files this notebook uses**

| File | What it does |
|---|---|
| `baselines/lstm_model.py` | The model: word embeddings, one bidirectional LSTM layer, a six-way output layer, and the vocabulary and batching helpers |
| `baselines/train.py` | Training, early stopping, evaluation and saving the results (`python -m baselines.train`) |
| `tests/test_lstm.py` | Automated tests for the model and the training helpers |
| `common/` | Shared label order, data loader, test subset and scoring used by every model |
| `results/lstm.json`, `results/lstm_preds.csv` | The files this run writes (metrics, parameters, time, cost; one prediction per test row) |

The run happens inside Colab's own copy of the repo, so it does not change the saved results in GitHub. Download the zip in the last cell to keep them.

In [ ]:
# Step 1: download the project code into a folder called "repo".
!git clone https://github.com/arunchowdary-1/Fake-News-Detection-Fine-tuned-Transformers-vs-Zero-shot-LLMs.git repo
# Move into the repo so the python -m commands below can find the baselines/ and common/ packages.
# (Keep comments on their own line: a trailing # after %cd would be read as part of the folder name.)
%cd repo
# Step 2: install what Colab does not have. Colab already includes torch and numpy.
# 'datasets<4' is pinned because datasets 4 no longer supports the script-based LIAR dataset.
!pip install -q 'datasets<4' scikit-learn
# Step 3: read LIAR from the TSV files that are already in the repo (liar_data/). On Colab the Hugging Face
# download of LIAR fails, and baselines/train.py has no fallback, so without this line the training cells stop
# with "Failed to load LIAR dataset". The loader uses this folder whenever LIAR_TSV_DIR is set.
%env LIAR_TSV_DIR=liar_data

In [ ]:
# Smoke test: 1 epoch only, to check that data loading, training, evaluation and saving all run without errors.
# Its scores are low and mean nothing. It writes results/lstm.json inside Colab's copy, and the full run below overwrites it.
!python -m baselines.train --epochs 1

In [ ]:
# Full run with the settings used for the reported results: up to 15 epochs, early stopping after 3 epochs
# without improvement on the validation macro-F1, batch size 64, learning rate 0.001, class-weighted loss, seed 42.
# Writes results/lstm.json, results/lstm_preds.csv and checkpoints/lstm_checkpoint.pt (weights are never committed).
!python -m baselines.train

In [ ]:
# Show the main numbers from the run. The reported run had accuracy 0.190, macro-F1 0.173 and 932,998 parameters
# on the shared 300-row test subset (small differences between machines are normal).
import json
r = json.load(open("results/lstm.json", encoding="utf-8"))
for k in ["accuracy", "macro_f1", "params", "train_minutes", "avg_latency_ms", "est_cost_per_1k", "n_eval"]:
    print(f"{k:16}", r[k])
print("full test set:", r["full_test"]["accuracy"], r["full_test"]["macro_f1"])

In [ ]:
# Run the automated tests for the BiLSTM (no GPU or download needed).
!python -m pytest tests/test_lstm.py -q

In [ ]:
# Bundle the two result files into one zip and download it to your computer.
# -j keeps only the file names, so the zip has no results/ folder inside.
from google.colab import files
!zip -j lstm_results.zip results/lstm.json results/lstm_preds.csv
files.download("lstm_results.zip")